In [1]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

import os
# os.environ["JAX_DISABLE_XLA"] = "0"
# XLA_PYTHON_CLIENT_PREALLOCATE=false
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"



def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [2]:
# @title 载入绘图函数

# 这定义了一个名为 select 的函数，它接受一个 xarray.Dataset 对象、一个指定数据集中变量的字符串，以及可选的水平和最大时间步数参数。它返回一个 xarray.Dataset。
def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
#     从数据集中选择与指定变量相对应的数据。
  data = data[variable]
#     如果数据集有一个名为 “batch” 的维度，这行代码选择数据的第一批。
  if "batch" in data.dims:
    data = data.isel(batch=0)
#     如果指定了 max_steps，并且数据集有一个 “time” 维度且步数多于 max_steps，这行代码将数据集限制在前 max_steps 个时间步。
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
#     如果指定了 level 并且它存在于数据集的坐标中，这行代码选择在该特定水平上的数据。
  if level is not None and "level" in data.coords:
    data = data.sel(level=level, method="nearest")
  return data

# 这定义了一个名为 scale 的函数，它接受一个 xarray.Dataset、一个用于缩放的可选中心值和一个表示是否使用鲁棒缩放的布尔值。
# 它返回一个包含数据集、matplotlib.colors.Normalize 对象和颜色映射名称的元组。
def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
#     这些行计算用于归一化的最小和最大值。如果 robust 为 True，它使用第2和第98百分位数来忽略异常值；否则，它使用绝对最小和最大值。
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
# 如果提供了 center 值，这些行调整 vmin 和 vmax 使其与中心等距，确保中心值是颜色刻度的中点。
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
#     函数返回数据集、配置有 vmin 和 vmax 的 Normalize 对象，以及要使用的颜色映射名称（如果有中心点则为 “RdBu_r”，否则为 “viridis”）。
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

# 这定义了一个名为 plot_data 的函数，它接受一个标题和 xarray.Dataset 对象的字典、图形标题、可选的绘图大小、鲁棒缩放标志和子图布局的列数。
# 它返回一个类似于 scale 函数的元组。
def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

# 这些行获取字典中的第一个数据集以确定时间步数（max_steps），并断言所有数据集都有相同数量的时间步。
  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

#     列数设置为指定列数或数据字典长度的最小值。根据列数计算行数。
  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
# 创建一个新的图形，其大小基于行数和列数以及指定的绘图大小。
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
#     设置图形的标题，并调整布局以消除子图之间的空白。
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

#     这个循环为字典中的每个数据集创建子图，设置轴和标题。
  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
#     每个子图使用 imshow 函数显示数据集的第一个时间步，使用指定的归一化和颜色映射。
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
#     为每个子图添加一个颜色条。
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

#     这定义了一个用于动画的 update 函数，它会在每个帧更新标题和数据。
  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

#     使用 FuncAnimation 类创建一个动画，它将为每个帧调用 update 函数。
  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
#     关闭图形（以防止它立即显示），并将动画转换为使用 JavaScript 的 HTML5 视频，然后返回。
  plt.close(figure.number)
  return HTML(ani.to_jshtml())

In [3]:
# @title 选择模型
# Rewrite by S.F. Sune, https://github.com/sfsun67.
'''
    我们有三种训练好的模型可供选择, 需要从https://console.cloud.google.com/storage/browser/dm_graphcast准备：
    GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
    GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz
    GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
'''
# 在此路径 /root/data/params 中查找结果，并列出 "params/"中所有文件的名称，去掉名称中的 "params/"perfix。

import os
import glob

# 定义数据目录，请替换成你自己的目录。
dir_path_params = "/root/code/GraphCast-from-Ground-Zero"


# Use glob to get all file paths in the directory
# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
file_paths_params = glob.glob(os.path.join(dir_path_params, "*"))

# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
# Remove the directory path and the ".../params/" prefix from each file name
params_file_options = [os.path.basename(path) for path in file_paths_params]

# 创建一个整数滑块控件random_mesh_size，用于选择网格大小。其默认值为4，最小值为4，最大值为6。较大的网格可以捕获更细的空间特征，但也会增加计算成本。
random_mesh_size = widgets.IntSlider(
    value=6, min=6, max=6, description="Mesh size:")
# 创建一个整数滑块控件random_gnn_msg_steps，用于选择GNN消息传递的步数。其默认值为4，最小值为1，最大值为32。较多的步数能让每个节点接收到来自更远邻居的信息，但会增加计算量和训练时间。
random_gnn_msg_steps = widgets.IntSlider(
    value=8, min=1, max=64, description="GNN message steps:")
# 创建一个下拉菜单控件random_latent_size，用于选择潜在大小。选项是2的4次方到2的9次方，即16到512，其默认值为32。较大的 latent_size 能够捕获更多的信息，但也可能导致过拟合或增加计算成本。
random_latent_size = widgets.Dropdown(
    options=[int(2**i) for i in range(4, 10)], value=16,description="Latent size:")
# 创建一个下拉菜单控件random_levels，用于选择压力水平。选项有13和37，其默认值为13。
random_levels = widgets.Dropdown(
    options=[3, 37], value=3, description="Pressure levels:")

# 创建一个下拉菜单控件params_file，用于选择参数文件。选项是之前从目录中获取的文件名列表。
params_file = widgets.Dropdown(
    options=params_file_options,
    description="Params file:",
    layout={"width": "max-content"})

# 创建一个标签页控件source_tab，其中包含两个标签。第一个标签是一个垂直布局的盒子，包含了前面创建的四个控件。第二个标签是参数文件的下拉菜单。
source_tab = widgets.Tab([
    widgets.VBox([
        random_mesh_size,
        random_gnn_msg_steps,
        random_latent_size,
        random_levels,
    ]),
    params_file,
])
# 为source_tab的两个标签设置标题，分别为"随机参数权重"和"预训练权重"。
source_tab.set_title(0, "随机参数权重（Random）")
source_tab.set_title(1, "预训练权重（Checkpoint）")
# 最后，创建一个垂直布局的盒子，包含了source_tab和一个标签，提示用户运行下一个单元格以加载模型，并告知重新运行该单元格将清除他们的选择。
widgets.VBox([
    source_tab,
    widgets.Label(value="运行下一个单元格以加载模型。重新运行该单元格将清除您的选择。")
])


In [4]:
# @title 加载模型

# 这行代码获取当前选中的标签页的标题，以确定用户选择了哪种参数权重（随机参数权重或预训练权重）。
source = source_tab.get_title(source_tab.selected_index)

# 如果用户选择了“随机参数权重”，则执行以下代码块。
if source == "随机参数权重（Random）":
#     初始化params为None和state为一个空字典。这些将在后面的代码中使用
  params = None  # Filled in below
  state = {}
# 创建一个model_config对象，它包含模型配置的参数。这些参数包括：
# resolution：分辨率，这里设置为0。
# mesh_size：网格大小，取自之前创建的滑块控件的值。
# latent_size：潜在大小，取自下拉菜单控件的值。
# gnn_msg_steps：GNN消息传递步数，取自滑块控件的值。
# hidden_layers：隐藏层的数量，这里设置为1。
# radius_query_fraction_edge_length：查询半径与边长的比例，这里设置为0.6。
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.6)
#     创建一个task_config对象，它包含任务配置的参数。这些参数包括：
# input_variables：输入变量。
# target_variables：目标变量。
# forcing_variables：强迫变量。
# pressure_levels：压力水平，取自下拉菜单控件的值。
# input_duration：输入持续时间。
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
#     如果用户选择了“预训练权重”，则执行以下代码块。
# 这段被注释的代码原本用于从Google Cloud Storage加载预训练权重。
# 现在，它被替换为从本地文件系统加载预训练权重的代码。使用open函数以二进制读取模式打开参数文件，并使用checkpoint.load函数加载检查点。
else:
  assert source == "预训练权重（Checkpoint）"
  '''with gcs_bucket.blob(f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)'''
  
  with open(f"{dir_path_params}/{params_file.value}", "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
    
#  从检查点中提取参数到params变量，并重新初始化state为一个空字典。 
  params = ckpt.params
  state = {}

# 从检查点中提取模型配置和任务配置。
  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("模型描述:\n", ckpt.description, "\n")
  print("模型许可信息:\n", ckpt.license, "\n")

model_config

ModelConfig(resolution=0, mesh_size=6, latent_size=16, gnn_msg_steps=8, hidden_layers=1, radius_query_fraction_edge_length=0.6, mesh2grid_edge_normalization_factor=None, patch_size_lat=8, patch_size_lon=8)

In [5]:
import xarray as xr
import numpy as np

dir_path_data = "/root/autodl-tmp/"
# 读取 NetCDF 文件到 example_batch
input_file_path = f"{dir_path_data}/example_batch.nc"
example_batch =  xr.open_dataset(input_file_path, chunks={'batch': 1})
# 关键修复：如果 batch 被保存成了坐标变量，就删掉它
if "batch" in example_batch.coords:
    example_batch = example_batch.drop_vars("batch")
# example_batch.load()

example_batch


<xarray.Dataset>
Dimensions:   (batch: 60, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * level     (level) float32 0.494 2.646 5.078
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>

In [6]:
# import os
# import glob
# import xarray as xr
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"

# # 读取所有训练文件
# file_paths = sorted(glob.glob(os.path.join(dir_path_data, "example_batch_*.nc")))

# if len(file_paths) == 0:
#     raise FileNotFoundError(f"在 {dir_path_data} 下没有找到 example_batch_*.nc 文件")

# print(f"共找到 {len(file_paths)} 个训练文件：")
# for p in file_paths:
#     print(" -", os.path.basename(p))

# # 只预览第一个文件，确认结构
# input_file_path = file_paths[0]
# print(f"\n预览文件：{os.path.basename(input_file_path)}")

# example_batch = xr.open_dataset(input_file_path, chunks={"batch": 1})
# example_batch

In [7]:
# 创建一个新的变量 land_sea_mask
import xarray as xr
import numpy as np

# 创建 land_sea_mask 变量，避免中间副本
example_batch["land_sea_mask"] = (
    (~np.isnan(example_batch["so"].isel(time=0, level=0)))
    .astype(np.float32)
)

example_batch = example_batch.fillna(0)
example_batch = example_batch.astype(np.float32)
example_batch

<xarray.Dataset>
Dimensions:        (batch: 60, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * level          (level) float32 0.494 2.646 5.078
  * lon            (lon) float32 -180.0 -179.9 -179.8 ... 179.8 179.8 179.9
  * lat            (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time           (time) timedelta64[ns] 0 days 1 days ... 10 days 11 days
    datetime       (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
Dimensions without coordinates: batch
Data variables:
    siconc         (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick        (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao         (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    land_sea_mask  (batch, lat, lon) float32 dask.array<chunksize=(1, 2041, 4320), meta=np.ndarray>

In [8]:
# # 创建一个新的变量 land_sea_mask，并封装成单文件加载函数
# import xarray as xr
# import numpy as np

# def load_one_file(file_path, batch_chunk=1):
#     ds = xr.open_dataset(file_path, chunks={"batch": batch_chunk})

#     # 创建 land_sea_mask 变量
#     ds["land_sea_mask"] = (
#         (~np.isnan(ds["so"].isel(time=0, level=0)))
#         .astype(np.float32)
#     )

#     # 缺失值填 0
#     ds = ds.fillna(0)

#     # 只转换数据变量到 float32，避免不必要的额外开销
#     for var in ds.data_vars:
#         if ds[var].dtype != np.float32:
#             ds[var] = ds[var].astype(np.float32)

#     return ds

# # # 加载第一个文件，供后续单步检查 / 调试使用
# # if "example_batch" in globals():
# #     try:
# #         example_batch.close()
# #     except Exception:
# #         pass

# # example_batch = load_one_file(file_paths[0], batch_chunk=1)
# # print(f"已加载示例文件：{file_paths[0]}")
# # example_batch

In [9]:
# @title 加载规范化数据
# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
import xarray
import os

dir_path_stats = "/root/data/stats/"

# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
# with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
#   diffs_stddev_by_level = xarray.load_dataset(f).compute()
# 类似于第4行，这行代码打开了另一个文件stats-mean_by_level.nc。
with open(f"{dir_path_stats}/stats-mean_by_level.nc", "rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
# 再次类似于第4行，这行代码打开了第三个文件stats-stddev_by_level.nc。
with open(f"{dir_path_stats}/stats-stddev_by_level.nc", "rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()
# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()

# 1）对 land_sea_mask 不做标准化：把 std 设为 1，均值设为 0
if "land_sea_mask" in stddev_by_level:
    stddev_by_level["land_sea_mask"] = xr.full_like(
        stddev_by_level["land_sea_mask"], 1.0
    )

if "land_sea_mask" in mean_by_level:
    mean_by_level["land_sea_mask"] = xr.full_like(
        mean_by_level["land_sea_mask"], 0.0
    )

def inspect_stats(ds: xarray.Dataset, name: str):
    print(f"\n==== {name} ====")
    print("dims:", ds.dims)
    print("vars:", list(ds.data_vars))
    for v in ds.data_vars:
        da = ds[v]
        print(
            f"{v}: shape={da.shape}, "
            f"min={float(da.min())}, "
            f"max={float(da.max())}, "
            f"mean={float(da.mean())}"
        )

inspect_stats(mean_by_level, "mean_by_level")
inspect_stats(stddev_by_level, "stddev_by_level")
inspect_stats(diffs_stddev_by_level, "diffs_stddev_by_level")



==== mean_by_level ====
dims: Frozen({'level': 17})
vars: ['so', 'thetao', 'uo', 'vo', 'siconc', 'sithick', 'usi', 'vsi', 'zos', 'land_sea_mask', 'year_progress_sin', 'year_progress_cos', 'day_progress_sin', 'day_progress_cos']
so: shape=(17,), min=34.16366958618164, max=34.85478973388672, mean=34.471527099609375
thetao: shape=(17,), min=6.003704071044922, max=14.141005516052246, mean=12.168696403503418
uo: shape=(17,), min=-0.0043333000503480434, max=0.011082032695412636, mean=0.004761594347655773
vo: shape=(17,), min=0.0005350023275241256, max=0.005490194074809551, mean=0.0033269007690250874
siconc: shape=(), min=0.8282931447029114, max=0.8282931447029114, mean=0.8282931447029114
sithick: shape=(), min=1.4046043157577515, max=1.4046043157577515, mean=1.4046043157577515
usi: shape=(), min=-0.01063524279743433, max=-0.01063524279743433, mean=-0.01063524279743433
vsi: shape=(), min=-0.000540390086825937, max=-0.000540390086825937, mean=-0.000540390086825937
zos: shape=(), min=-0.142186

In [10]:
# @title Build jitted functions, and possibly initialize random weights
# Construct the model and initialize the weights.
# 构建模型并初始化权重

# 模型组网
# Construct the model
def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # Deeper one-step predictor.
  predictor = graphcast.GraphCast(model_config, task_config)

  # Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to
  # from/to float32 to/from BFloat16.
  predictor = casting.Bfloat16Cast(predictor)

  # Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from
  # BFloat16 happens after applying normalization to the inputs/targets.
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # Wraps everything so the one-step model can produce trajectories.
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor

# 前向运算
# forward
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

# 计算损失函数
# loss function
@hk.transform_with_state    # used to convert a pure function into a stateful function
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)    # constructs and wraps a GraphCast Predictor, which is a model used for making predictions in a graph-based machine learning task.
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

# 计算梯度
# gradient
def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), model_config, task_config,
        i, t, f)
    return loss, (diagnostics, next_state)
  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads

# Jax doesn't seem to like passing configs as args through the jit. Passing it
# in via partial (instead of capture by closure) forces jax to invalidate the
# jit cache if you change configs.
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# Always pass params and state, so the usage below are simpler
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# Our models aren't stateful, so the state is always empty, so just return the
# predictions. This is requiredy by our rollout code, and generally simpler.
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

init_jitted = jax.jit(with_configs(run_forward.init))
loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
    run_forward.apply))))

In [11]:
# from graphcast import normalization
# from graphcast import xarray_tree

# wrapper = normalization.InputsAndResiduals(
#     predictor=None,  # 这里只用它的内部函数，不真的跑网络
#     stddev_by_level=stddev_by_level,
#     mean_by_level=mean_by_level,
#     diffs_stddev_by_level=diffs_stddev_by_level,
# )

# train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0, 1)) for var in example_batch.data_vars})
#         # train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0,1)) for var in example_batch.data_vars})
# train_batch.load()
# print(train_batch.dims.mapping)
# train_steps = 1
        
# train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
#             train_batch,
#             target_lead_times=slice("24h", f"{train_steps * 24}h"),
#             **dataclasses.asdict(task_config))

# # 取一个 batch
# b0_inputs = train_inputs.isel(batch=0, drop=False)
# b0_targets = train_targets.isel(batch=0, drop=False)
# b0_forcings = train_forcings.isel(batch=0, drop=False)

# # 看归一化后的 target residual
# norm_target_residuals = xarray_tree.map_structure(
#     lambda t: wrapper._subtract_input_and_normalize_target(b0_inputs, t),
#     b0_targets,
# )

# print("\n=== normalized target residual stats ===")
# for v in norm_target_residuals.data_vars:
#     da = norm_target_residuals[v]
#     print(
#         f"{v}: min={float(da.min())}, "
#         f"max={float(da.max())}, "
#         f"mean={float(da.mean())}"
#     )


In [12]:
import numpy as np
import optax  # 导入 optax 库
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import gc  # 导入垃圾回收模块
from jax import device_put
from jax import devices

def load_params_from_npz(filepath):
    # 加载 .npz 文件，启用 allow_pickle
    loaded = np.load(filepath, allow_pickle=True)
    
    # 将加载的内容转换为字典
    params = {key: loaded[key].item() if loaded[key].dtype == object else loaded[key] for key in loaded.files}
    
    print(f"模型参数已从 {filepath} 加载")
    return params

# 定义文件路径
load_filepath = "model_params_all_2.npz"

# 调用加载函数，读取参数
params = load_params_from_npz(load_filepath)


optimizer = optax.adam(learning_rate=0.0001)
opt_state = optimizer.init(params)

模型参数已从 model_params_all_2.npz 加载


In [13]:
from pathlib import Path
import numpy as np
import jax
import orbax.checkpoint as ocp

ckpt_dir = Path("/root/autodl-tmp/graphcast_ckpts")
ckpt_dir.mkdir(parents=True, exist_ok=True)

ckpt_options = ocp.CheckpointManagerOptions(
    max_to_keep=3,
    save_interval_steps=1,
)

ckpt_manager = ocp.CheckpointManager(
    ckpt_dir,
    options=ckpt_options,
)

def save_ckpt(ckpt_manager, epoch, params, opt_state, state, train_steps):
    train_state = {
        "params": params,
        "opt_state": opt_state,
        "state": state,
        "epoch": np.array(epoch, dtype=np.int32),
        "train_steps": np.array(train_steps, dtype=np.int32),
    }

    ckpt_manager.save(
        epoch,
        args=ocp.args.StandardSave(train_state),
    )
    
def restore_ckpt(ckpt_manager, step, params, opt_state, state):
    abstract_train_state = {
        "params": jax.tree_util.tree_map(ocp.utils.to_shape_dtype_struct, params),
        "opt_state": jax.tree_util.tree_map(ocp.utils.to_shape_dtype_struct, opt_state),
        "state": jax.tree_util.tree_map(ocp.utils.to_shape_dtype_struct, state),
        "epoch": np.array(0, dtype=np.int32),
        "train_steps": np.array(0, dtype=np.int32),
    }

    restored = ckpt_manager.restore(
        step,
        args=ocp.args.StandardRestore(abstract_train_state),
    )
    return restored

In [14]:
import optax  # 导入 optax 库
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import gc  # 导入垃圾回收模块
from jax import device_put
from jax import devices


latest_step = ckpt_manager.latest_step()
print("latest_step =", latest_step)

restored = restore_ckpt(
    ckpt_manager=ckpt_manager,
    step=latest_step,
    params=params,
    opt_state=opt_state,
    state=state,
)

params = restored["params"]
opt_state = restored["opt_state"]
state = restored["state"]

print("恢复完成")


# def release_memory(*var_names):
#     for var_name in var_names:
#         if var_name in globals():  # 检查变量是否存在于全局命名空间
#             del globals()[var_name]
#     gc.collect()
    
def release_local(*vars_to_del):
    """删除本地大对象引用，触发 GC。"""
    for v in vars_to_del:
        try:
            del v
        except NameError:
            pass
    gc.collect()

# release_memory("train_batch")

start_epoch = 26
total_epochs = 30
learning_rate = 0.0001

# 关键：train_steps 要对齐到“epoch 18 开始前”的状态
# 你的规则是：epoch=3,6,9,12,15 时各加 1
# 所以 epoch 18 开始前，train_steps 应该是 6
train_steps = 1 + max(0, (start_epoch - 1) // 3)

batch_size = example_batch.dims["batch"]

optimizer = optax.adam(learning_rate=learning_rate)

# 关键：不要重新 optimizer.init(params) 覆盖旧的 opt_state
# 只有第一次训练、opt_state 还不存在时才初始化
if opt_state is None:
    opt_state = optimizer.init(params)

def print_grads(grads):
    for key, value in grads.items():
        if isinstance(value, dict):
            print_grads(value)
        else:
            print(f"{key}: {jnp.mean(jnp.abs(value)):.6f}")
    
# 训练循环
for epoch in range(start_epoch, total_epochs):
    # 每5轮增加1个时间步长
    if epoch % 3 == 0 and epoch > 0:
        train_steps += 1
        print(f"训练步长增加: 当前 train_steps = {train_steps}")
    print(f"当前 Epoch: {epoch}, Train Steps: {train_steps}")
    
    for start_idx in range(batch_size):
        train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(start_idx, start_idx + 1)) for var in example_batch.data_vars})
        # train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0,1)) for var in example_batch.data_vars})
        train_batch.load()
        # print(train_batch.dims.mapping)
        
        train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
            train_batch,
            target_lead_times=slice("24h", f"{train_steps * 24}h"),
            **dataclasses.asdict(task_config))
        
        # print(train_inputs.dims.mapping)
        # # 打印train_inputs的维度和数据类型
        # print("train_inputs变量维度：")
        # for var_name, var_data in train_inputs.items():
        #     print(f"Variable: {var_name}, Shape: {var_data.shape}, dtype: {var_data.dtype}")
        del train_batch
        gc.collect()
        print("释放内存")
        # 如果参数为空，则初始化参数和状态
        if params is None:
          params, state = init_jitted(
              rng=jax.random.PRNGKey(0),
              inputs=train_inputs,
              targets_template=train_targets,
              forcings=train_forcings)
          opt_state = optimizer.init(params)  # 确保用初始化后的 params 初始化 opt_state
            
        if params is None:
            print("params is None")
        if state is None:
            print("state is None")
        
        # 在每个批次中使用不同的随机数种子
        rng = random.PRNGKey(epoch * total_epochs + start_idx)
        # print("batch start")
        
        loss, diagnostics, next_state, grads = grads_fn_jitted(
            params=params,
            state=state,
            inputs=train_inputs,
            targets=train_targets,
            forcings=train_forcings
        )
        state = next_state
        # print("compute grads")

        # 使用优化器更新参数
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)  # 应用更新到参数
        # 只在当前 epoch 的最后一个 batch 打印一次梯度信息
        if start_idx == batch_size - 1:
            mean_grad = np.mean(
                jax.tree_util.tree_flatten(
                    jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads)
                )[0]
            )
            print(
                f"Epoch {epoch}, Last batch {start_idx}: "
                f"Loss: {loss:.4f}  Mean |grad|: {mean_grad:.6f}"
            )
        del train_inputs, train_targets, train_forcings, diagnostics, grads
        gc.collect()
        # mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])   
        # print(f"Epoch {epoch}: Loss: {loss:.4f} :Mean |grad|: {mean_grad:.6f}")
        # 打印各层梯度均值
        # print_grads(grads)
        
    jax.clear_caches()

latest_step = 27
恢复完成
当前 Epoch: 26, Train Steps: 9
释放内存
[Patch-Grid] num_grid_nodes = 8817120, num_patch_nodes = 138240, patch_size_lat = 8, patch_size_lon = 8
Patch-Grid2Mesh edges: 216800
[Grid2Mesh] num_grid_nodes = 138240, num_edges = 216800, neighbors per node: min=1, mean=1.57, max=3
Mesh2Grid Edges: 17634240
stacked_array.sizes: Frozen({'batch': 1, 'lat': 2041, 'lon': 4320, 'channels': 14})
template_dataset.sizes: Frozen({'time': 1, 'batch': 1, 'lat': 2041, 'lon': 4320, 'level': 3})
stacked_array: <xarray.Variable (batch: 1, lat: 2041, lon: 4320, channels: 14)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[1,2041,4320,14])>with<DynamicJaxprTrace(level=5/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (time: 1, batch: 1, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * time     (time) timedelta64[ns] 1 days
  * lon      (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat      (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.75 89.83 89.92 90.0
  *

2026-04-14 15:21:58.317612: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 1s:

  %scatter.322 = bf16[138240]{0} scatter(bf16[138240]{0} %broadcast.279, s32[8817120,1]{1,0} %constant.282, bf16[8817120,1]{1,0} %broadcast.2320), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_0.318, metadata={op_name="jit(<unnamed wrapped function>)/jit(main)/patch_pool_mlp/scatter-add[update_consts=() dimension_numbers=ScatterDimensionNumbers(update_window_dims=(), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,)) indices_are_sorted=False unique_indices=False mode=GatherScatterMode.FILL_OR_DROP]" source_file="/root/code/code/GraphCast-from-Ground-Zero/graphcast/graphcast.py" source_line=362}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding fr

释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
Epoch 26, Last batch 59: Loss: 16.9479  Mean |grad|: 0.020207
训练步长增加: 当前 train_steps = 10
当前 Epoch: 27, Train Steps: 10
释放内存
[Patch-Grid] num_grid_nodes = 8817120, num_patch_nodes = 138240, patch_size_lat = 8, patch_size_lon = 8
Patch-Grid2Mesh edges: 216800
[Grid2Mesh] num_grid_nodes = 138240, num_edges = 216800, neighbors per node: min=1, mean=1.57, max=3
Mesh2Grid Edges: 17634240
stacked_array.sizes: Frozen({'batch': 1, 'lat': 2041, 'lon': 4320, 'channels': 14})
template_dataset.sizes: Frozen({'time': 1, 'batch': 1, 'lat': 2041, 'lon': 4320, 'level': 3})
stacked_array: <xarray.Variable (batch: 1, lat: 2041, lon: 4320, channels: 14)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[1,2041

2026-04-14 16:00:45.009560: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 2s:

  %scatter.322 = bf16[138240]{0} scatter(bf16[138240]{0} %broadcast.279, s32[8817120,1]{1,0} %constant.282, bf16[8817120,1]{1,0} %broadcast.2320), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_0.318, metadata={op_name="jit(<unnamed wrapped function>)/jit(main)/patch_pool_mlp/scatter-add[update_consts=() dimension_numbers=ScatterDimensionNumbers(update_window_dims=(), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,)) indices_are_sorted=False unique_indices=False mode=GatherScatterMode.FILL_OR_DROP]" source_file="/root/code/code/GraphCast-from-Ground-Zero/graphcast/graphcast.py" source_line=362}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding fr

释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
Epoch 27, Last batch 59: Loss: 18.3906  Mean |grad|: 0.021218
当前 Epoch: 28, Train Steps: 10
释放内存
[Patch-Grid] num_grid_nodes = 8817120, num_patch_nodes = 138240, patch_size_lat = 8, patch_size_lon = 8
Patch-Grid2Mesh edges: 216800
[Grid2Mesh] num_grid_nodes = 138240, num_edges = 216800, neighbors per node: min=1, mean=1.57, max=3
Mesh2Grid Edges: 17634240
stacked_array.sizes: Frozen({'batch': 1, 'lat': 2041, 'lon': 4320, 'channels': 14})
template_dataset.sizes: Frozen({'time': 1, 'batch': 1, 'lat': 2041, 'lon': 4320, 'level': 3})
stacked_array: <xarray.Variable (batch: 1, lat: 2041, lon: 4320, channels: 14)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[1,2041,4320,14])>with<DynamicJaxpr

2026-04-14 16:35:08.804176: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 4s:

  %scatter.322 = bf16[138240]{0} scatter(bf16[138240]{0} %broadcast.279, s32[8817120,1]{1,0} %constant.282, bf16[8817120,1]{1,0} %broadcast.2320), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_0.318, metadata={op_name="jit(<unnamed wrapped function>)/jit(main)/patch_pool_mlp/scatter-add[update_consts=() dimension_numbers=ScatterDimensionNumbers(update_window_dims=(), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,)) indices_are_sorted=False unique_indices=False mode=GatherScatterMode.FILL_OR_DROP]" source_file="/root/code/code/GraphCast-from-Ground-Zero/graphcast/graphcast.py" source_line=362}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding fr

释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
Epoch 28, Last batch 59: Loss: 18.3906  Mean |grad|: 0.026174
当前 Epoch: 29, Train Steps: 10
释放内存
[Patch-Grid] num_grid_nodes = 8817120, num_patch_nodes = 138240, patch_size_lat = 8, patch_size_lon = 8
Patch-Grid2Mesh edges: 216800
[Grid2Mesh] num_grid_nodes = 138240, num_edges = 216800, neighbors per node: min=1, mean=1.57, max=3
Mesh2Grid Edges: 17634240
stacked_array.sizes: Frozen({'batch': 1, 'lat': 2041, 'lon': 4320, 'channels': 14})
template_dataset.sizes: Frozen({'time': 1, 'batch': 1, 'lat': 2041, 'lon': 4320, 'level': 3})
stacked_array: <xarray.Variable (batch: 1, lat: 2041, lon: 4320, channels: 14)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[1,2041,4320,14])>with<DynamicJaxpr

2026-04-14 16:54:50.544219: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 8s:

  %scatter.322 = bf16[138240]{0} scatter(bf16[138240]{0} %broadcast.279, s32[8817120,1]{1,0} %constant.282, bf16[8817120,1]{1,0} %broadcast.2320), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_0.318, metadata={op_name="jit(<unnamed wrapped function>)/jit(main)/patch_pool_mlp/scatter-add[update_consts=() dimension_numbers=ScatterDimensionNumbers(update_window_dims=(), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,)) indices_are_sorted=False unique_indices=False mode=GatherScatterMode.FILL_OR_DROP]" source_file="/root/code/code/GraphCast-from-Ground-Zero/graphcast/graphcast.py" source_line=362}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding fr

释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
释放内存
Epoch 29, Last batch 59: Loss: 18.3344  Mean |grad|: 0.023730


In [15]:
# import os
# import gc
# import random
# import optax
# import jax
# import jax.numpy as jnp
# from jax import random as jrandom
# import numpy as np
# import xarray as xr


# def release_local(*vars_to_del):
#     """删除本地大对象引用，触发 GC。"""
#     for v in vars_to_del:
#         try:
#             del v
#         except NameError:
#             pass
#     gc.collect()


# # 初始化训练步长
# train_steps = 1

# # 训练轮数和学习率
# total_epochs = 100
# learning_rate = 0.0001

# # 定义优化器
# optimizer = optax.adam(learning_rate=learning_rate)
# opt_state = optimizer.init(params) if params is not None else None


# def print_grads(grads):
#     for key, value in grads.items():
#         if isinstance(value, dict):
#             print_grads(value)
#         else:
#             print(f"{key}: {jnp.mean(jnp.abs(value)):.6f}")


# # 训练循环：epoch -> file -> batch
# for epoch in range(total_epochs):
#     if epoch % 10 == 0 and epoch > 0:
#         train_steps += 1
#         print(f"训练步长增加: 当前 train_steps = {train_steps}")

#     print(f"\n当前 Epoch: {epoch}, Train Steps: {train_steps}")

#     # # 每个 epoch 打乱文件顺序
#     # epoch_files = file_paths.copy()
#     # random.shuffle(epoch_files)
#     epoch_files = file_paths

#     for file_idx, file_path in enumerate(epoch_files):
#         print(f"\n[Epoch {epoch}] 正在训练文件 {file_idx + 1}/{len(epoch_files)}: {os.path.basename(file_path)}")

#         # 读取并预处理当前文件
#         example_batch = load_one_file(file_path, batch_chunk=1)
#         file_batch_size = example_batch.sizes["batch"]

#         # 打乱当前文件内部的 batch 顺序
#         # batch_indices = np.random.permutation(file_batch_size)
#         batch_indices = range(file_batch_size)

#         for step_in_file, start_idx in enumerate(batch_indices):
#             train_batch = xr.Dataset({
#                 var: example_batch[var].isel(batch=slice(start_idx, start_idx + 1))
#                 for var in example_batch.data_vars
#             })

#             # 真正把这一小批数据加载到内存
#             train_batch.load()

#             train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
#                 train_batch,
#                 target_lead_times=slice("24h", f"{train_steps * 24}h"),
#                 **dataclasses.asdict(task_config)
#             )

#             del train_batch
#             gc.collect()

#             # 如果参数为空，则初始化参数和状态
#             if params is None:
#                 params, state = init_jitted(
#                     rng=jax.random.PRNGKey(0),
#                     inputs=train_inputs,
#                     targets_template=train_targets,
#                     forcings=train_forcings
#                 )
#                 opt_state = optimizer.init(params)

#             if params is None:
#                 print("params is None")
#             if state is None:
#                 print("state is None")

#             # 每个 batch 使用不同随机种子
#             rng = jrandom.PRNGKey(epoch * 100000 + file_idx * 10000 + int(start_idx))

#             loss, diagnostics, next_state, grads = grads_fn_jitted(
#                 params=params,
#                 state=state,
#                 inputs=train_inputs,
#                 targets=train_targets,
#                 forcings=train_forcings
#             )
#             state = next_state

#             # 参数更新
#             updates, opt_state = optimizer.update(grads, opt_state, params)
#             params = optax.apply_updates(params, updates)

#             # 每个文件的最后一个 batch 打印一次
#             if step_in_file == file_batch_size - 1:
#                 mean_grad = np.mean(
#                     jax.tree_util.tree_flatten(
#                         jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads)
#                     )[0]
#                 )
#                 print(
#                     f"Epoch {epoch}, File {file_idx + 1}/{len(epoch_files)}, "
#                     f"Last batch {start_idx}: Loss: {loss:.4f}  Mean |grad|: {mean_grad:.6f}"
#                 )

#             del train_inputs, train_targets, train_forcings, diagnostics, grads
#             gc.collect()

#         # 当前文件训练完，关闭文件句柄并释放内存
#         try:
#             example_batch.close() 
#         except Exception:
#             pass

#         # del example_batch
#         gc.collect()
#         # jax.clear_caches()

In [1]:
import numpy as np

# 假设 'params' 是你训练后的模型参数
# 你可以通过以下代码保存到 .npz 文件

def save_params_to_npz(params, filepath):
    # 将params的内容转换为字典结构，便于保存
    params_dict = {k: v for k, v in params.items()}
    
    # 使用np.savez将参数保存到指定的文件中
    np.savez(filepath, **params_dict)
    print(f"模型参数已保存到 {filepath}")

# 定义文件保存路径
save_filepath = "model_params.npz"

# 调用保存函数，将参数保存
save_params_to_npz(params, save_filepath)

NameError: name 'params' is not defined

In [ ]:


save_ckpt(
        ckpt_manager=ckpt_manager,
        epoch=epoch,
        params=params,
        opt_state=opt_state,
        state=state,
        train_steps=train_steps,
    )